In [ ]:
# name: pyopensky_client
# date: 27/07/2026
# author: Toby Alexander

# description: 
# Query historical FLARM data from the OpenSky Network's Trino database (https://openskynetwork.github.io/opensky-api/trino.html)
# Includes further parsing and filtering of the data stream to examine spatial distribution of messages and identify unique flights.

# required modules:
#   - Python 3.6+ (created with 3.10.19)
#   - matplotlib
#   - numpy
#   - pandas
#   - pyopensky (https://pypi.org/project/pyopensky/)
#   - rs1090 (https://pypi.org/project/rs1090/)
#   - sqlalchemy (https://pypi.org/project/SQLAlchemy/)

In [ ]:
import os

# Set credentials in a temporary environment variable for the current session. This is a workaround for the pyopensky tool not being able to read the credentials from the config file.
os.environ['OPENSKY_USERNAME'] = ''
os.environ['OPENSKY_PASSWORD'] = ''

from pyopensky.config import opensky_config_dir     # access to opensky config directory
from sqlalchemy import select                      # for constructing SQL queries
from sqlalchemy import text
from pyopensky.schema import FlarmRaw               # for accessing the FlarmRaw table in the database
from pyopensky.trino import Trino                   # for connecting to the Trino database
from datetime import datetime                       # for working with timestamps
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
import rs1090                                       # for decoding raw messages

print(opensky_config_dir)                           # location of config file where credentials are stored.
# The tool seems to be unsuccessful in reading the credentials from this config file, hence why the credentials are instead set in an envir var.

# 1. Construct and send Trino query

In [ ]:
trino = Trino()             # activate Trino client

# df = []

# Important to make queries as efficient as possible to reduce server load and speed up querying
# Select specific columns of interest to reduce size of data transfer
# ALWAYS constrain search by hour. The dataset is organised into hour partitions.

# Sample of sounding locations to centre bounding box around.

# Essen 10410
lat0 = 51.404 
long0 = 6.968

# Meppen 10304
# lat0 = 52.715 
# long0 = 7.318

# # Bergen, Germany 10238
# lat0 = 52.815 
# long0 = 9.925

# # De Bilt, Netherlands 06260
# lat0 = 52.100
# long0 = 5.177

# Set range of bounding box in degrees of lat and long.
rangelat = 1
rangelong = 1

# Construct SQL query to select messages within bounding box and time range, and with correct CRC to remove erroneous messages.
# hour constraints are defined directly in the query. Saw issues with using variables for the hour constraints, so they are hardcoded here.
query = (
    select(FlarmRaw)
    .with_only_columns(
        FlarmRaw.sensorlatitude,
        FlarmRaw.sensorlongitude,
        FlarmRaw.sensoraltitude,
        FlarmRaw.timestamp,
        FlarmRaw.rawmessage,
        FlarmRaw.hour,
    )
    .where(FlarmRaw.hour >= "2023-06-04 09:00:00")
    .where(FlarmRaw.hour < "2023-06-04 13:00:00")
    .where(FlarmRaw.sensorlatitude > lat0 - rangelat)
    .where(FlarmRaw.sensorlatitude < lat0 + rangelat)
    .where(FlarmRaw.sensorlongitude > long0 - rangelong)
    .where(FlarmRaw.sensorlongitude < long0 + rangelong)
    .where(FlarmRaw.crccorrect == True)
    .order_by(FlarmRaw.timestamp)
)

df = trino.query(query)     # make query as above, and store in dataframe
df = df.reset_index(drop=True)

In [ ]:
df.head()

In [ ]:
# df.to_parquet("flarm_data.parquet")

In [ ]:
# # Read saved parquet file back in at a later time

# root = ""
# filename = ''   # replace date
# filepath = root + filename

# df = pd.read_parquet("flarm_data.parquet")


In [ ]:
df.info()

# 2. Decode rawmessage using rs1090 tool

In [ ]:
# Decode raw messages contained in rawmessage column of trino database rows.
# This allows us to access the icao24 part of the message, which identifies unique flight IDs
# This uses Python tool rs1090.
# You need to pass in certain parameters to the tool in order to decode successfully. These are included in the trino database

# an if statement handles unsuccessful decoding, which can sometimes happen. 
# It keeps track of the indices of successfully decoded rows.
# This is important because we want to merge the dataframe of decoded data from the raw message with the rest of the trino dataframe.
# And the two dataframes can have different numbers of rows if some messages were not able to be decoded.

decoded = []
successful_indices = []

for i, row in df.iterrows():
    try:
        result = rs1090.flarm(
            row['rawmessage'],
            int(row['timestamp']),
            row['sensorlatitude'],
            row['sensorlongitude']
        )
        if result is not None:
            result['original_index'] = i  # Keep track of which row it came from
            decoded.append(result)
            successful_indices.append(i)
    except Exception as e:
        print(f"Failed to decode message at index {i}: {e}")

# Create decoded DataFrame
decoded_dfm = pd.DataFrame(decoded)

# Get only the successfully decoded rows from original DataFrame
original_subset = df.loc[successful_indices]

# Reset indices so they align
original_subset = original_subset.reset_index(drop=True)
decoded_dfm = decoded_dfm.reset_index(drop=True)

# DEBUG to check whether there were any issues with the decoding process.
print(f"decoded length: {len(decoded)}")
print(f"successful_indices length: {len(successful_indices)}")
print(f"original_subset shape: {original_subset.shape}")
print(f"decoded_dfm shape: {decoded_dfm.shape}")

In [ ]:
# Rename the timestamp column. We are merging with the original trino dataframe that also has a timestamp column.
# We want to keep the two distinct. The timestamp of the rawmessage has a resolution of 1s only, so we use the trino timestamp instead.
decoded_dfm = decoded_dfm.rename(columns={'timestamp': 'raw_timestamp'})

# Merge (concatenate) the columns you want
merged_dfm = pd.concat([
    original_subset[['timestamp']],  # columns from original
    decoded_dfm
], axis=1)

In [ ]:
merged_dfm.head()

In [ ]:
# We get a greater resolution with the timestamps from the trino database than from the raw message, which truncates to the nearest second.
# The rs1090 decoder must be truncating the timestamp. This cell shows an example of the difference between the two timestamps. The difference is always less than 1 second, which is expected.

print(f"Trino truncated timestamp:{merged_dfm['timestamp'][2]}")
print(f"rs1090 decoded timestamp:{merged_dfm['raw_timestamp'][2]:.3f}")

print("Alignment check (should be 0):")
np.sum(abs(merged_dfm['timestamp'] - merged_dfm['raw_timestamp']) > 1)    # Logic to check that the timestamps are aligned. Should be 0.

In [ ]:
# Filter the DataFrame to only include messages decoded as gliders

gliders_df = merged_dfm[merged_dfm['actype'] == 'Glider']
decoded_gliders_df = gliders_df.reset_index(drop=True)
print(f"glider messages: {len(gliders_df)}")
print(f"fraction decoded as gliders: {len(gliders_df) / len(merged_dfm):.2%}")

In [ ]:
# plot reference latitude and longitude to check airfield location of flights
plt.figure(figsize=(4, 3))
plt.scatter(gliders_df.reference_lon, gliders_df.reference_lat, s=50, alpha=0.5)
plt.plot(long0, lat0, 'ro', label='Reference Location')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.title('Reference Locations of Decoded Messages')
plt.legend()
plt.show()

In [ ]:
# plot message latitude and longitude to check message distribution
plt.figure(figsize=(4, 3))
plt.scatter(gliders_df.longitude, gliders_df.latitude, s=50, alpha=0.5)
plt.plot(long0, lat0, 'ro', label='Reference Location')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.title('Locations of Decoded Messages')
plt.xlim([long0-1.5,long0+1.5])
plt.ylim([lat0-1.5,lat0+1.5])
plt.legend()
plt.show()

In [ ]:
def latlong_to_xy_equirect(lat, lon, lat0, long0):
    R = 6.378137*1e6                     # mean Earth radius in meters
    lat_rad = np.radians(lat)
    lon_rad = np.radians(lon)
    lat0_rad = np.radians(lat0)
    lon0_rad = np.radians(long0)
    x = R * (lon_rad - lon0_rad) * np.cos(lat0_rad)
    y = R * (lat_rad - lat0_rad)

    return x, y

x, y = latlong_to_xy_equirect(gliders_df.latitude, gliders_df.longitude, lat0, long0)

# plot projected latitude and longitude to check separation between messages and reference sounding.
plt.figure(figsize=(4, 3))
plt.scatter(x/1e3, y/1e3, s=50, alpha=0.5)
plt.plot(0, 0, 'ro', label='Reference Location')
plt.xlabel('x (km)')
plt.ylabel('y (km)')
plt.title('Locations of Decoded Messages')
plt.xlim([-150,150])
plt.ylim([-150,150])
plt.legend()
plt.show()

In [ ]:
# Save the DataFrame to a Parquet file (optional)

gliders_df.to_parquet('trino_flarm_20230604_20230604_Essen_4pt0degsq.parquet', index=False)

# 3. Filter for proximity to reference sounding

In [ ]:
# Filter for messages that are within so many hours/metres of ref sounding time/location.
x, y = latlong_to_xy_equirect(gliders_df.latitude, gliders_df.longitude, lat0, long0)
mag = np.sqrt(x**2 + y**2)

time_err = 7200
disp_err = 5e4

ref_timestamp = datetime(2023, 6, 4, 11, 45, 0).timestamp()   # replace with ref sounding time
dts = abs(gliders_df['timestamp'] - ref_timestamp)    # time difference in seconds
ddisp = abs(mag)    # displacement from reference location in meters
gliders_df_filtered = gliders_df[(dts <= time_err) & (ddisp <= disp_err)] # keep only messages within 1 hour of ref sounding time
gliders_df_filtered = gliders_df_filtered.reset_index(drop=True)
print(f"Number of messages after filtering for time_err {time_err/3600} hours and disp_err {disp_err/1e3} km: {len(gliders_df_filtered)}")

In [ ]:
x, y = latlong_to_xy_equirect(gliders_df_filtered.latitude, gliders_df_filtered.longitude, lat0, long0)

# plot projected latitude and longitude of filtered messages.
plt.figure(figsize=(4, 3))
plt.scatter(x/1e3, y/1e3, s=50, alpha=0.5)
plt.plot(0, 0, 'ro', label='Reference Location')
plt.xlabel('x (km)')
plt.ylabel('y (km)')
plt.title('Locations of Filtered Messages')
plt.xlim([-1.5*disp_err/1e3,1.5*disp_err/1e3])
plt.ylim([-1.5*disp_err/1e3,1.5*disp_err/1e3])
plt.legend()
plt.show()

# 4. Group messages into individual flights

In [ ]:
# Group the filtered glider messages by aircraft (using the icao24 identifier) to identify uniqueq flights, and store in a dictionary 'unq_f'.

unq_f = {}      # unique flights
grouped = gliders_df_filtered.groupby('icao24')      # Group rows in the DataFrame by the 'icao24' column, which identifies individual aircraft

for icao, group in grouped:
    unq_f[icao] = group    

In [ ]:
print(f"num of unique aircraft: {len(unq_f.keys())}")
unq_f.keys()
nmes = np.empty(len(unq_f.keys()))
keys = np.empty(len(unq_f.keys()), dtype='U32')  # Use string dtype for keys

i = 0
for key in unq_f.keys():
    nmes[i], keys[i] = len(unq_f[key]), key
    i += 1

# Sort keys and nmes together in descending order of nmes
sorted_indices = np.argsort(nmes)[::-1]     # Get indices that would sort
nmes = nmes[sorted_indices]                 # Sort nmes
keys = keys[sorted_indices]                 # Sort keys using the same indices

print(f"number of flights with > 2000 messages: {np.sum(nmes > 2000)}")

In [ ]:
# Work with the most active gliders (the one with the most messages) for further analysis
keyidx = 0  # Select a key
gkey = keys[keyidx]     # Get the icao24 identifier for the selected glider
print(f"key: {gkey}, number of messages: {nmes[keyidx]}")

g1df = unq_f[gkey]  # Copy to a new variable for further analysis. This is the DataFrame of messages for the selected glider.

In [ ]:
# A couple of errors in the data (geospatial coords far from main cluster) suggest some messages are not decoded correctly, so filter these out
print(f"before filtering, size of g1df: {len(g1df)}")
g1df_lat_mean = g1df.latitude.mean()
g1df_lon_mean = g1df.longitude.mean()
g1df_lat_std = g1df.latitude.std()
g1df_lon_std = g1df.longitude.std()

# Filter out messages with longitude and latitude values that are more than 4 standard deviations from the mean
g1df = g1df[(g1df.longitude > g1df_lon_mean - 4 * g1df_lon_std) & (g1df.longitude < g1df_lon_mean + 4 * g1df_lon_std) & (g1df.latitude > g1df_lat_mean - 4 * g1df_lat_std) & (g1df.latitude < g1df_lat_mean + 4 * g1df_lat_std)]
print(f"after filtering, size of g1df: {len(g1df)}")

In [ ]:
# Plot out the lat and lon to check the flight path of this glider

t0 = datetime.fromtimestamp(g1df.timestamp.iloc[0]).strftime('%Y-%m-%d %H:%M:%S')
t1 = datetime.fromtimestamp(g1df.timestamp.iloc[-1]).strftime('%Y-%m-%d %H:%M:%S')

plt.figure(figsize=(8, 5))

plt.plot(g1df.longitude, g1df.latitude)

plt.xlabel("longitude, deg")
plt.ylabel("latitude, deg")
plt.suptitle(f"Flight path for Glider with ICAO24: {gkey}")
plt.title(f"timestamp: {t0} to {t1}")

In [ ]:
# Find the time interval between each message and the previous one, to check for any large gaps in the data

type(g1df)
dts = np.diff(g1df.timestamp)
print(np.shape(dts))
g1df['dts'] = np.concatenate(([np.nan], dts), dtype=float)  # taking diff reduces the length by 1, so concatenate a NaN at the start to keep the same length as the original DataFrame
# print(g1df[['timestamp', 'dts']].head())

In [ ]:
# Plot the time intervals between messages to check for any large gaps in the data

plt.figure(figsize=(10, 6))
plt.hist(g1df['dts'], bins=100, range=[-0.5, 4.5], color='blue')
plt.xlabel('Time interval (seconds)')
plt.ylabel('Frequency')
plt.title('Distribution of time intervals between messages')
plt.suptitle(f"ICAO24: {gkey}")
plt.grid(True)
plt.show()

In [ ]:
print(f"Proportion of time steps with dt >= 10: {100*np.sum(dts>=10)/len(dts):.2f}%")

In [ ]:
# Save out to a parquet file for later analysis.

filename = f"flarm_raw_{gkey}.parquet"

g1df.to_parquet(filename, index=False)
g1df.to_csv(filename.replace('.parquet', '.csv'), index=False)